<a href="https://colab.research.google.com/github/samlekum/CTT-Forecasting-Expanding/blob/main/numberplateocr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A. Persiapan Data
  ## 1. Setup Kaggle dan cek disk

In [2]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

!pip install -q -U kaggle
!df -h /content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 7.3 MB/s eta 0:00:00
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  43% /


## 2. Cek dan download dataset

In [5]:
!kaggle datasets files binh234/ccpd2019

name                    size  creationDate                
---------------  -----------  --------------------------  
CCPD2019.tar.xz  13164924944  2023-11-27 05:05:34.806000  


In [6]:
!kaggle datasets download -d binh234/ccpd2019 -p /content/data --unzip

Dataset URL: https://www.kaggle.com/datasets/binh234/ccpd2019
License(s): CC0-1.0
100% 12.3G/12.3G [02:45<00:00, 79.7MB/s]



In [7]:
!ls -lh /content/data

total 13G
-rw-r--r-- 1 root root 13G Sep 21 16:41 CCPD2019.tar.xz


## 3. Ekstrak Dataset

In [8]:
!tar -xJf /content/data/CCPD2019.tar.xz -C /content/data

In [9]:
!rm /content/data/CCPD2019.tar.xz
!du -sh /content/data

25G	/content/data


## 4. Cek struktur dataset

In [10]:
!ls /content/data
!ls /content/data/CCPD2019

CCPD2019
ccpd_base  ccpd_challenge  ccpd_fn  ccpd_rotate  ccpd_weather  README.md
ccpd_blur  ccpd_db	   ccpd_np  ccpd_tilt	 LICENSE       splits


In [11]:
!for d in /content/data/CCPD2019/*/; do echo "$d: $(ls $d | wc -l)"; done

/content/data/CCPD2019/ccpd_base/: 199996
/content/data/CCPD2019/ccpd_blur/: 20611
/content/data/CCPD2019/ccpd_challenge/: 50003
/content/data/CCPD2019/ccpd_db/: 10132
/content/data/CCPD2019/ccpd_fn/: 20967
/content/data/CCPD2019/ccpd_np/: 3036
/content/data/CCPD2019/ccpd_rotate/: 10053
/content/data/CCPD2019/ccpd_tilt/: 30216
/content/data/CCPD2019/ccpd_weather/: 9999
/content/data/CCPD2019/splits/: 9


In [12]:
!ls /content/data/CCPD2019/splits
!head -3 /content/data/CCPD2019/splits/train.txt

ccpd_blur.txt	    ccpd_db.txt  ccpd_rotate.txt  test.txt   val.txt
ccpd_challenge.txt  ccpd_fn.txt  ccpd_tilt.txt	  train.txt
ccpd_base/0092816091954-94_82-181&490_358&548-363&554_189&540_190&484_364&498-0_0_28_29_16_29_32-133-13.jpg
ccpd_base/0104418103448-91_84-329&442_511&520-515&519_340&508_326&447_501&458-0_0_33_18_25_26_26-166-27.jpg
ccpd_base/023275862069-90_86-173&473_468&557-485&563_189&555_187&469_483&477-0_0_2_27_9_26_24-178-36.jpg


## 5. Parser label plat nomor

In [13]:
import os

PROVINCES = ["皖","沪","津","渝","冀","晋","蒙","辽","吉","黑","苏","浙","京","闽","赣","鲁","豫","鄂","湘","粤","桂","琼","川","贵","云","藏","陕","甘","青","宁","新","警","学","O"]
ALPHABETS = list("ABCDEFGHJKLMNPQRSTUVWXYZ") + ["O"]
ADS = list("ABCDEFGHJKLMNPQRSTUVWXYZ0123456789") + ["O"]

def parse_ccpd(path):
    name = os.path.splitext(os.path.basename(path))[0]
    f = name.split('-')
    (x1, y1), (x2, y2) = [tuple(map(int, p.split('&'))) for p in f[2].split('_')]
    idx = list(map(int, f[4].split('_')))
    plate = PROVINCES[idx[0]] + ALPHABETS[idx[1]] + ''.join(ADS[i] for i in idx[2:])
    return {'bbox': (x1, y1, x2, y2), 'plate': plate}

In [14]:
import glob, cv2
from google.colab.patches import cv2_imshow

files = sorted(glob.glob('/content/data/CCPD2019/ccpd_base/*.jpg'))[:3]
for p in files:
    print(parse_ccpd(p))

x1, y1, x2, y2 = parse_ccpd(files[0])['bbox']
img = cv2.imread(files[0])
cv2_imshow(img[y1:y2, x1:x2])

{'bbox': (352, 516, 448, 547), 'plate': '皖AYL250'}
{'bbox': (283, 519, 381, 553), 'plate': '皖AH2T95'}
{'bbox': (441, 517, 538, 546), 'plate': '皖APS969'}


## 6. Pre-crop plat ke .npz

In [15]:
!wc -l /content/data/CCPD2019/splits/*.txt

   20611 /content/data/CCPD2019/splits/ccpd_blur.txt
   50003 /content/data/CCPD2019/splits/ccpd_challenge.txt
   10132 /content/data/CCPD2019/splits/ccpd_db.txt
   20967 /content/data/CCPD2019/splits/ccpd_fn.txt
   10053 /content/data/CCPD2019/splits/ccpd_rotate.txt
   30216 /content/data/CCPD2019/splits/ccpd_tilt.txt
  141982 /content/data/CCPD2019/splits/test.txt
  100000 /content/data/CCPD2019/splits/train.txt
   99996 /content/data/CCPD2019/splits/val.txt
  483960 total


In [16]:
import numpy as np, cv2, time
from multiprocessing import Pool

ROOT = '/content/data/CCPD2019'
W, H = 94, 24
os.makedirs('/content/cache', exist_ok=True)

def load_split(name):
    with open(f'{ROOT}/splits/{name}.txt') as f:
        return [l.strip() for l in f if l.strip()]

def process(rel):
    p = f'{ROOT}/{rel}'
    info = parse_ccpd(p)
    x1, y1, x2, y2 = info['bbox']
    img = cv2.imread(p)
    crop = cv2.resize(img[max(0, y1):y2, max(0, x1):x2], (W, H))
    return crop, info['plate']

def build(name):
    t = time.time()
    rels = load_split(name)
    with Pool(2) as pool:
        out = pool.map(process, rels, chunksize=256)
    X = np.stack([o[0] for o in out])
    y = np.array([o[1] for o in out])
    np.savez(f'/content/cache/{name}.npz', X=X, y=y)
    print(name, X.shape, f'{time.time()-t:.0f}s')

build('val')

val (99996, 24, 94, 3) 322s


In [17]:
build('train')
build('test')

train (100000, 24, 94, 3) 324s
test (141982, 24, 94, 3) 385s


## 7. Backup ke Google Drive

In [18]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ccpd_cache
!cp /content/cache/*.npz /content/drive/MyDrive/ccpd_cache/
!du -sh /content/drive/MyDrive/ccpd_cache

Mounted at /content/drive
2.2G	/content/drive/MyDrive/ccpd_cache


# B. Training Model
## 8. Load data & charset

In [19]:
import numpy as np, torch, torch.nn as nn, time
from google.colab import drive
drive.mount('/content/drive')

D = '/content/drive/MyDrive/ccpd_cache'
tr, va, te = [np.load(f'{D}/{n}.npz') for n in ('train', 'val', 'test')]
Xtr, ytr = tr['X'], tr['y']
Xva, yva = va['X'], va['y']
Xte, yte = te['X'], te['y']

chars = sorted(set(''.join(ytr)) | set(''.join(yva)) | set(''.join(yte)))
c2i = {c: i + 1 for i, c in enumerate(chars)}   # 0 = blank CTC
i2c = {i: c for c, i in c2i.items()}
assert all(len(s) == 7 for y in (ytr, yva, yte) for s in y), 'ada plat yang panjangnya bukan 7'
print('jumlah karakter:', len(chars))
print('train/val/test :', Xtr.shape, Xva.shape, Xte.shape)
print('contoh label   :', ytr[:3])

dev = 'cuda'
def to_gpu(X): return torch.from_numpy(X).to(dev).permute(0, 3, 1, 2).contiguous()  # uint8, N,3,H,W
Xtr_g, Xva_g, Xte_g = to_gpu(Xtr), to_gpu(Xva), to_gpu(Xte)
ytr_t = torch.tensor([[c2i[c] for c in s] for s in ytr]).to(dev)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
jumlah karakter: 65
train/val/test : (100000, 24, 94, 3) (99996, 24, 94, 3) (141982, 24, 94, 3)
contoh label   : ['皖A45S58' '皖A9U122' '皖AC3K20']


## 9. Model CRNN + fungsi evaluasi

In [24]:
def prep(x): return (x.float() / 255 - 0.5) / 0.5

class CRNN(nn.Module):
    def __init__(s, nc):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
        s.cnn = nn.Sequential(
            blk(3, 64), nn.MaxPool2d(2),
            blk(64, 128), nn.MaxPool2d(2),
            blk(128, 256), blk(256, 256), nn.MaxPool2d((2, 1)),
            blk(256, 256), nn.MaxPool2d((3, 1)))
        s.rnn = nn.LSTM(256, 128, num_layers=2, bidirectional=True, batch_first=True)
        s.fc = nn.Linear(256, nc)
    def forward(s, x):
        f = s.cnn(x).squeeze(2).permute(0, 2, 1)   # B, T=23, 256
        o, _ = s.rnn(f)
        return s.fc(o)

model = CRNN(len(chars) + 1).to(dev)
ctc = nn.CTCLoss(blank=0, zero_infinity=True)

def decode(out):
    res = []
    for r in out.argmax(2).cpu().numpy():
        s, prev = [], 0
        for k in r:
            if k != prev and k != 0: s.append(i2c[k])
            prev = k
        res.append(''.join(s))
    return res

@torch.no_grad()
def evaluate(X, y, bs=1024):
    model.eval(); ok = 0
    for i in range(0, len(X), bs):
        pred = decode(model(prep(X[i:i+bs])))
        ok += sum(a == b for a, b in zip(pred, y[i:i+bs]))
    return ok / len(X)

print(sum(p.numel() for p in model.parameters()) / 1e6, 'juta parameter')

2.359426 juta parameter


## 10. Training

In [25]:
EPOCHS, BS = 15, 256   # tes dulu 2 epoch; kalau lancar, ganti EPOCHS jadi 15 lalu jalanin ulang
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
steps = len(Xtr_g) // BS
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPOCHS * steps)
best = 0

for ep in range(EPOCHS):
    model.train(); t = time.time()
    tot = torch.zeros((), device=dev)
    perm = torch.randperm(len(Xtr_g), device=dev)
    for i in range(steps):
        idx = perm[i*BS:(i+1)*BS]
        out = model(prep(Xtr_g[idx]))
        lp = out.log_softmax(2).permute(1, 0, 2)
        il = torch.full((len(idx),), out.size(1), dtype=torch.long)
        tl = torch.full((len(idx),), 7, dtype=torch.long)
        loss = ctc(lp, ytr_t[idx], il, tl)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5)
        opt.step(); sched.step(); tot += loss.detach()
    acc = evaluate(Xva_g, yva)
    print(f'ep {ep+1}/{EPOCHS} loss {tot.item()/steps:.3f} val_acc {acc:.4f} {time.time()-t:.0f}s')
    if acc > best:
        best = acc
        torch.save({'model': model.state_dict(), 'chars': chars}, '/content/drive/MyDrive/ccpd_cache/crnn_best.pt')

ep 1/15 loss 3.560 val_acc 0.0000 47s
ep 2/15 loss 0.688 val_acc 0.9398 49s
ep 3/15 loss 0.032 val_acc 0.9586 49s
ep 4/15 loss 0.019 val_acc 0.9661 49s
ep 5/15 loss 0.012 val_acc 0.9659 49s
ep 6/15 loss 0.007 val_acc 0.9666 49s
ep 7/15 loss 0.006 val_acc 0.9791 49s
ep 8/15 loss 0.004 val_acc 0.9872 49s
ep 9/15 loss 0.002 val_acc 0.9906 49s
ep 10/15 loss 0.001 val_acc 0.9923 49s
ep 11/15 loss 0.001 val_acc 0.9939 49s
ep 12/15 loss 0.000 val_acc 0.9942 49s
ep 13/15 loss 0.000 val_acc 0.9942 49s
ep 14/15 loss 0.000 val_acc 0.9943 49s
ep 15/15 loss 0.000 val_acc 0.9943 49s


## 11. Evaluasi di test set

In [26]:
ckpt = torch.load('/content/drive/MyDrive/ccpd_cache/crnn_best.pt', map_location=dev)
model.load_state_dict(ckpt['model'])
print('val acc :', evaluate(Xva_g, yva))
print('test acc:', evaluate(Xte_g, yte))

model.eval()
with torch.no_grad():
    pred = decode(model(prep(Xte_g[:2000])))
salah = [(a, b) for a, b in zip(pred, yte[:2000]) if a != b]
print(len(salah), 'salah dari 2000; contoh (prediksi, label):', salah[:10])

val acc : 0.9942697707908317
test acc: 0.5209040582609064
1082 salah dari 2000; contoh (prediksi, label): [('皖AL981N', np.str_('皖AE981N')), ('皖A57F14', np.str_('皖AS7F14')), ('皖AF4079', np.str_('皖AF601V')), ('皖AX441', np.str_('皖A7L443')), ('皖A4D606', np.str_('皖A4D806')), ('皖A197L8', np.str_('皖A187L8')), ('皖AW6E16', np.str_('皖AW8E16')), ('皖A991', np.str_('皖A891Y2')), ('皖A4J99', np.str_('皖AJ4J99')), ('皖AZQ166', np.str_('皖AZQ966'))]


## 12. Akurasi per subset di test set (diagnosa)

In [27]:
rels = [l.strip() for l in open('/content/data/CCPD2019/splits/test.txt') if l.strip()]
sub = np.array([r.split('/')[0] for r in rels])

model.eval(); preds = []
with torch.no_grad():
    for i in range(0, len(Xte_g), 1024):
        preds += decode(model(prep(Xte_g[i:i+1024])))
preds = np.array(preds)

for s in np.unique(sub):
    m = sub == s
    print(f'{s:16s} n={m.sum():6d} acc={(preds[m] == yte[m]).mean():.4f}')

ccpd_blur        n= 20611 acc=0.4631
ccpd_challenge   n= 50003 acc=0.6360
ccpd_db          n= 10132 acc=0.4426
ccpd_fn          n= 20967 acc=0.5860
ccpd_rotate      n= 10053 acc=0.4952
ccpd_tilt        n= 30216 acc=0.3596
